needs major updating (see k-means notebooks)

# Mixture of Gaussians

Testing the mixture of gaussians clustering algorithm which is able to find clusters with hyper-ellipssoid shape.

entropy, kurtosis, crest factor.

test c-means and gaussian

test total_energy calculation, clean up code.

feature selection + numerical studies
(for k-means and c-means, which set of metrics works the best for each.)

1st paper is k-means
2nd paper compares k-means with c-means and gaussian.

then write up and focus on the second paper. dicuss different classification methods

how to test wavelet orders? better representation of transients vs computational cost and cluster separation.

In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
%matplotlib inline
sys.path.append("..")

import wavelet_transform.utils.training_data as train
import wavelet_transform.utils.bubble_data_manager as bdm
import wavelet_transform.utils.wavelet_features as extract
import wavelet_transform.postprocess.plotting as plot
from wavelet_transform.utils.machine_learning import train_mixture_of_gaussians
from wavelet_transform.utils.preprocessing import preprocess

### Training data

Run bubble dynamics over a range of frequencies and intensities to generate a larger array of training data.

In [ ]:
# Set sampling rate and noise std
sampling_rate = 50e6
std_noise = 0.01

# Set length of data to keep
sample_time = 20 / 2e6
data_len = int(sample_time * sampling_rate)

# Parameters for the wavelet transform
wavelet = "db10"
levels = 8
mode = "symmetric"
use_min_entropy = True
min_level = 4
freq_threshold = 20e6 / (sampling_rate / 2)
window_data = False

In [ ]:
wpt_df = train.get_training_data(
    wavelet, sampling_rate, levels, mode, std_noise, 
    use_min_entropy, min_level, freq_threshold, window_data
)

### Training

Train the mixture of gaussians on the generated data

In [ ]:
# Get data
classifying_metrics = [
    "average energy",
    "kurtosis",
    # "crest factor",
    # "log average energy",
    # "level entropy",
    "spectral entropy",
    # "skewness"
]

# Parameters for mixture of gaussians
n_clusters = 5

# Train model
results = train_mixture_of_gaussians(wpt_df, classifying_metrics, n_clusters)
data_train, labels, data_train_normalised, centers, scaler,  gm = results

centers = pd.DataFrame(centers, columns=classifying_metrics)
centers_sorted, labels_sorted = plot.relabel(labels, centers, "average energy")

In [ ]:
fig = plot.plot_clustering(data_train, centers_sorted, labels_sorted, classifying_metrics)

# Plot centroid metrics
fig = plot.plot_centers(classifying_metrics, centers_sorted)

### Numerical studies

In [ ]:
wavelets = [f"db{i}" for i in range(4, 21)]

In [ ]:
from sklearn.metrics import silhouette_score
scores = []

for wavelet in wavelets:
    print(wavelet)
    wpt_df = train.get_training_data(
        wavelet, sampling_rate, levels, mode, std_noise, 
        use_min_entropy, min_level, freq_threshold, window_data
    )
    _, _labels, _data_train_normalised, *_   = train_mixture_of_gaussians(wpt_df, classifying_metrics, n_clusters)
    scores.append(silhouette_score(_data_train_normalised, _labels)) 
    
fig, ax = plt.subplots()
fig.set_size_inches(15, 7)
plt.bar(wavelets, scores)
plt.title("silhouette score vs db order")
plt.show()

### Classifying new data points

new labels may not match sorted labels

In [ ]:
frequency = 10E5
pac = -2E5
R_init = 7e-6

min_energy = 0  # Only nodes with energy at least min_energy * total_energy will be considered

data = bdm.get_data(R_init, frequency, pac, sampling_rate)
data = preprocess(data, window=window_data)

new_wpt_df, new_tr_obj = extract.extract_wavelet_features(
    data, 
    levels,
    wavelet, 
    mode,
    include_tr_obj=True,
    min_entropy=use_min_entropy,
    min_level=min_level,
    freq_threshold=freq_threshold
)

plt.plot(data)
plt.show()

In [ ]:
# new_tr_obj.calc_frequencies(sampling_rate)
# plot.heisenberg(new_tr_obj)

In [ ]:
# New testing data from signal
new_entry = new_wpt_df[classifying_metrics]

# Standardize the new entry using the same scaler
new_entry_normalized = scaler.transform(new_entry)

# New labels
new_labels = gm.predict(new_entry_normalized)

# Display the predicted results
print("Predicted hard cluster assignment:", labels)

In [ ]:
fig = plot.plot_reconstructions_by_group(new_tr_obj, new_labels)